# MIPO-NDD: training ETA

Open this in a **separate** Colab tab while the training notebook runs cell 2. A CPU runtime is enough (Runtime -> Change runtime type -> CPU). It only reads Drive.


In [ ]:
# ETA from real progress. Run in a SEPARATE Colab notebook (a free CPU runtime is enough): the training
# notebook is busy with cell 2. Reads Drive only.
import json, os, statistics, time
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
EXP = Path(os.environ.get('MIPO_EXP', '/content/drive/MyDrive/MIPO_NDD/v4_validation_mechanism_650M_bafe6e1c78fd218e'))
WINDOW_H = 3                     # measure speed over the last 3 hours
now = float(os.environ.get('MIPO_NOW', time.time()))

protocol = json.loads((EXP/'protocol.json').read_text())
runs = [EXP/'runs'/m/f'seed_{s}'/f'fold_{f:02d}' for m in protocol['models'] for s in protocol['seeds'] for f in protocol['folds']]
finished = [r for r in runs if (r/'test_summary.json').exists()]
per_run = [json.loads((r/'training_status.json').read_text())['epochs_run'] for r in finished] or [17]
avg_epochs = statistics.mean(per_run)
recent, gaps, left = [], [], 0.
for r in runs:
    stamps = sorted(p.stat().st_mtime for p in r.glob('validation_epochs/epoch_*.csv'))
    recent += [t for t in stamps if now-t < WINDOW_H*3600]
    gaps += [b-a for a, b in zip(stamps, stamps[1:]) if now-b < WINDOW_H*3600 and b-a < 3600]
    if r not in finished:
        left += max(avg_epochs-len(stamps), 1)
recent.sort()
print(f'{len(finished)}/{len(runs)} runs finished; about {left:.0f} epochs left (finished runs averaged {avg_epochs:.1f} epochs)')
if len(recent) < 3:
    print('Not enough epochs in the last few hours to measure speed yet; check again after ~30 minutes of training.')
else:
    rate = (len(recent)-1)/((recent[-1]-recent[0])/3600)
    print(f'Last {WINDOW_H} h: {rate:.1f} epochs/hour across all jobs; one epoch takes {statistics.median(gaps)/60:.1f} min per job (DGX: ~7)')
    eta_h = left/rate
    print(f'ETA: {eta_h:.0f} h of training (~{eta_h/24:.1f} days), finishing around '
          f"{time.strftime('%a %d %b %H:%M', time.localtime(now+eta_h*3600))} (runtime clock) if nothing stops it."
          ' Add ~15 min per Colab restart.')
